# 📘 YaqeenAI — Notebook 01: Data Preparation

**What this notebook does** (run on Google Colab):
1. Loads raw `dorar_hadith_full_dataset.json` from Google Drive
2. Cleans and normalizes Arabic text (strips tashkeel, tatweel, prefixes)
3. Detects hadith grade from the `ruling` field
4. Generates stable document IDs
5. Computes `canonical_group_id` for deduplication
6. Builds **metadata-enriched embedding text** (matn + rawi + masdar + grade)
7. Computes **dataset statistics** JSON (for "how many hadith?" queries)
8. Exports `cleaned_hadiths.parquet` + `dataset_stats.json` to Google Drive

**Output** → Google Drive:
- `cleaned_hadiths.parquet` — ready for notebook 02
- `dataset_stats.json` — downloaded to local `data/` for stats queries

In [ ]:
# Step 0: Install dependencies
!pip install -q pandas pyarrow pyarabic

In [ ]:
# Step 1: Mount Google Drive & load raw dataset
from google.colab import drive
drive.mount('/content/drive')

import os, json

# ⚠️ Change this to match YOUR Drive folder
DATA_DIR = '/content/drive/MyDrive/colabnotebooks/yaqeen_hadith_rag/data'
DATASET_FILE = f'{DATA_DIR}/dorar_hadith_full_dataset.json'

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(
        f"\n❌ File not found: {DATASET_FILE}\n"
        f"Upload 'dorar_hadith_full_dataset.json' to '{DATA_DIR}' in Google Drive first."
    )

with open(DATASET_FILE, 'r', encoding='utf-8') as f:
    all_hadiths = json.load(f)

print(f"✅ Loaded {len(all_hadiths):,} hadiths")
print(f"   File size: {os.path.getsize(DATASET_FILE) / (1024*1024):.1f} MB")
print(f"   Sample keys: {list(all_hadiths[0].keys())}")

In [ ]:
# Step 2: Convert to DataFrame & inspect
import pandas as pd

df = pd.DataFrame(all_hadiths)
del all_hadiths  # free memory

print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    print(f"  {col:30s}  non-null: {non_null:>7,}  ({non_null/len(df)*100:.1f}%)")

print(f"\n--- Sample row ---")
df.head(1).T

In [ ]:
# Step 3: Grade detection from the 'ruling' field
import re

# Map raw Arabic ruling text → canonical grade
GRADE_PATTERNS = [
    # Sahih (authentic)
    (re.compile(r'صحيح|ثابت|محفوظ|إسناده صحيح|على شرط الشيخين|على شرط البخاري|على شرط مسلم', re.IGNORECASE), 'sahih', 'صحيح'),
    # Hasan (good)
    (re.compile(r'حسن|جيد|لا بأس به|قوي', re.IGNORECASE), 'hasan', 'حسن'),
    # Daif (weak)
    (re.compile(r'ضعيف|لين|فيه مقال|إسناده ضعيف|لا يصح|منكر|غريب|شاذ', re.IGNORECASE), 'daif', 'ضعيف'),
    # Mawdu (fabricated)
    (re.compile(r'موضوع|باطل|لا أصل له|مكذوب|كذب', re.IGNORECASE), 'mawdu', 'موضوع'),
]

def detect_grade(ruling: str) -> tuple[str, str]:
    """Return (grade_en, grade_ar) from a ruling string."""
    if not ruling or not isinstance(ruling, str):
        return 'unknown', 'غير معروف'
    ruling = ruling.strip()
    for pattern, grade_en, grade_ar in GRADE_PATTERNS:
        if pattern.search(ruling):
            return grade_en, grade_ar
    return 'unknown', 'غير معروف'

# Apply
grades = df['ruling'].apply(detect_grade)
df['grade']    = grades.apply(lambda x: x[0])
df['grade_ar'] = grades.apply(lambda x: x[1])

print("Grade distribution:")
print(df['grade'].value_counts().to_string())
print(f"\nTotal: {len(df):,}")

In [ ]:
# Step 4: Generate stable document IDs & clean missing fields
import hashlib

def make_doc_id(row) -> str:
    """Deterministic ID: grade__hadith_tag (or grade__md5 if tag missing)."""
    tag = str(row.get('hadith_tag', '') or '').strip()
    grade = row.get('grade', 'unknown')
    if tag:
        return f"{grade}__{tag}"
    # Fallback: hash the hadith text
    text = str(row.get('hadith', '') or '').strip()
    h = hashlib.md5(text.encode('utf-8')).hexdigest()[:12]
    return f"{grade}__md5_{h}"

df['doc_id'] = df.apply(make_doc_id, axis=1)

# Fix duplicates by appending a counter
dup_mask = df.duplicated(subset='doc_id', keep=False)
if dup_mask.any():
    dup_count = 0
    seen = {}
    new_ids = df['doc_id'].tolist()
    for i, (is_dup, did) in enumerate(zip(dup_mask, new_ids)):
        if is_dup:
            seen[did] = seen.get(did, 0) + 1
            if seen[did] > 1:
                new_ids[i] = f"{did}__dup{seen[did]}"
                dup_count += 1
    df['doc_id'] = new_ids
    print(f"⚠️  Fixed {dup_count} duplicate doc_ids")

# Fill NaN for key text fields
for col in ['hadith', 'rawi', 'mohadeth', 'book', 'ruling']:
    df[col] = df[col].fillna('')

print(f"✅ {df['doc_id'].nunique():,} unique doc_ids out of {len(df):,} rows")
print(f"   Sample: {df['doc_id'].iloc[0]}")

In [ ]:
# Step 5: Arabic text normalization
import pyarabic.araby as araby

def normalize_arabic(text: str) -> str:
    """Remove tashkeel, normalize alef/taa, strip extra whitespace."""
    if not text:
        return ''
    # Remove diacritics (tashkeel)
    text = araby.strip_tashkeel(text)
    # Normalize hamzat alef forms → bare alef
    text = re.sub(r'[إأآا]', 'ا', text)
    # Normalize taa marbuta → haa
    text = text.replace('ة', 'ه')
    # Normalize alef maksura → yaa
    text = text.replace('ى', 'ي')
    # Remove tatweel (kashida)
    text = text.replace('ـ', '')
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# text_ar: normalized (for embedding & search)
# text_ar_raw: original with tashkeel (for display)
df['text_ar_raw'] = df['hadith'].str.strip()
df['text_ar']     = df['hadith'].apply(normalize_arabic)

# Normalize rawi too (for consistent matching)
df['rawi_normalized'] = df['rawi'].apply(normalize_arabic)

# Drop rows with empty hadith text
before = len(df)
df = df[df['text_ar'].str.len() > 10].reset_index(drop=True)
after = len(df)
print(f"✅ Normalized {after:,} hadiths ({before - after} empty/short rows removed)")
print(f"\n--- Sample ---")
print(f"Raw:        {df['text_ar_raw'].iloc[0][:120]}...")
print(f"Normalized: {df['text_ar'].iloc[0][:120]}...")

In [ ]:
# Step 6: Canonical group ID (deduplication key for near-identical hadiths)

def canonical_group_id(text: str) -> str:
    """
    Hash the first 80 chars of normalized text → group ID.
    Hadiths that share the same opening (same matn, different chains)
    get the same canonical_group_id for dedup at retrieval time.
    """
    if not text:
        return 'empty'
    prefix = text[:80]
    return hashlib.md5(prefix.encode('utf-8')).hexdigest()[:12]

df['canonical_group_id'] = df['text_ar'].apply(canonical_group_id)

n_groups = df['canonical_group_id'].nunique()
print(f"✅ {n_groups:,} canonical groups from {len(df):,} hadiths")
print(f"   Avg group size: {len(df)/n_groups:.1f}")
print(f"   Largest groups:")
print(df['canonical_group_id'].value_counts().head(5).to_string())

## Step 7: Build metadata-enriched embedding text

**Key design decision:** Instead of embedding only the hadith text (`text_ar`), we embed a metadata-enriched string that includes the narrator, source, and grade. This dramatically improves retrieval for queries like:
- *"أحاديث أبو هريرة"* (hadiths by Abu Hurayra)
- *"أحاديث صحيح البخاري"* (hadiths from Sahih Bukhari)
- *"الأحاديث الصحيحة عن الصبر"* (authentic hadiths about patience)

**Format:**
```
الراوي: أبو هريرة | المصدر: صحيح البخاري | الدرجة: صحيح | المتن: إنما الأعمال بالنيات...
```

In [ ]:
# Step 7: Build metadata-enriched embedding text

def build_embedding_text(row) -> str:
    """
    Combine metadata labels with the normalized hadith text.
    This is what gets embedded — so semantic search matches metadata too.
    """
    parts = []

    rawi = str(row.get('rawi', '') or '').strip()
    if rawi:
        parts.append(f"الراوي: {rawi}")

    masdar = str(row.get('book', '') or '').strip()
    if masdar:
        parts.append(f"المصدر: {masdar}")

    grade_ar = str(row.get('grade_ar', '') or '').strip()
    if grade_ar and grade_ar != 'غير معروف':
        parts.append(f"الدرجة: {grade_ar}")

    matn = str(row.get('text_ar', '') or '').strip()
    parts.append(f"المتن: {matn}")

    return ' | '.join(parts)

df['embedding_text'] = df.apply(build_embedding_text, axis=1)

# Quick stats
avg_len = df['embedding_text'].str.len().mean()
max_len = df['embedding_text'].str.len().max()
print(f"✅ Built embedding_text for {len(df):,} hadiths")
print(f"   Avg length: {avg_len:.0f} chars | Max: {max_len:,} chars")
print(f"\n--- Sample embedding_text ---")
print(df['embedding_text'].iloc[0][:300])

In [ ]:
# Step 8: Compute dataset statistics (for "how many" questions)

stats = {
    "total_hadiths": len(df),

    # By grade
    "by_grade": df['grade'].value_counts().to_dict(),
    "by_grade_ar": df['grade_ar'].value_counts().to_dict(),

    # Top narrators
    "top_narrators": (
        df[df['rawi'].str.strip() != '']['rawi']
        .value_counts().head(30).to_dict()
    ),
    "unique_narrators": int(df[df['rawi'].str.strip() != '']['rawi'].nunique()),

    # Top sources (books)
    "top_sources": (
        df[df['book'].str.strip() != '']['book']
        .value_counts().head(30).to_dict()
    ),
    "unique_sources": int(df[df['book'].str.strip() != '']['book'].nunique()),

    # Top muhadditheen
    "top_muhadditheen": (
        df[df['mohadeth'].str.strip() != '']['mohadeth']
        .value_counts().head(20).to_dict()
    ),
    "unique_muhadditheen": int(df[df['mohadeth'].str.strip() != '']['mohadeth'].nunique()),

    # Categories
    "by_category": (
        df['hadith_tag'].value_counts().head(30).to_dict()
        if 'hadith_tag' in df.columns else {}
    ),

    # Explanation coverage
    "has_explanation_count": int(
        df['hasExplanation'].sum() if 'hasExplanation' in df.columns else 0
    ),
}

# Save to Drive
stats_path = f'{DATA_DIR}/dataset_stats.json'
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"✅ Saved dataset_stats.json → {stats_path}")
print(f"\n📊 Summary:")
print(f"   Total hadiths:       {stats['total_hadiths']:>10,}")
for grade, count in stats['by_grade'].items():
    print(f"   {grade:20s} {count:>10,}")
print(f"   Unique narrators:    {stats['unique_narrators']:>10,}")
print(f"   Unique sources:      {stats['unique_sources']:>10,}")
print(f"   Unique muhadditheen: {stats['unique_muhadditheen']:>10,}")
print(f"   With explanation:    {stats['has_explanation_count']:>10,}")

In [ ]:
# Step 9: Select final columns & export to Parquet

EXPORT_COLUMNS = [
    'doc_id',
    'text_ar',               # normalized (for search / sparse retrieval)
    'text_ar_raw',           # original with tashkeel (for display)
    'embedding_text',        # metadata-enriched (for dense embedding in notebook 02)
    'grade',                 # e.g. sahih, hasan, daif, mawdu, unknown
    'grade_ar',              # e.g. صحيح, حسن, ضعيف
    'ruling',                # raw ruling text
    'rawi',                  # narrator name (original)
    'rawi_normalized',       # narrator name (normalized)
    'mohadeth',              # muhaddith
    'book',                  # source book (masdar)
    'numberOrPage',          # page/number reference
    'hadith_tag',            # category tag
    'hasExplanation',        # boolean
    'canonical_group_id',    # dedup group key
]

# Keep only columns that exist
existing_cols = [c for c in EXPORT_COLUMNS if c in df.columns]
df_export = df[existing_cols].copy()

# Save parquet
parquet_path = f'{DATA_DIR}/cleaned_hadiths.parquet'
df_export.to_parquet(parquet_path, index=False, engine='pyarrow')

file_size = os.path.getsize(parquet_path) / (1024 * 1024)
print(f"✅ Exported {len(df_export):,} hadiths → {parquet_path}")
print(f"   File size: {file_size:.1f} MB")
print(f"   Columns:   {existing_cols}")
print(f"\n🎉 Notebook 01 complete! Proceed to notebook 02 for embedding & indexing.")